<a href="https://colab.research.google.com/github/subairkprm/Backup_replit_Spread_Verse/blob/main/Python_Data_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException, Depends, Header
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import pandas as pd
import pytesseract
from PIL import Image
import io
import json
import re
from typing import Dict, Any

# Initialize FastAPI App with Swagger Documentation
app = FastAPI(
    title="Elite Route Data Engine",
    description="Python Microservice for Advanced Data Processing (Pandas) and OCR (Tesseract)",
    version="1.0.0"
)

# Allow CORS for the Next.js frontend to communicate securely
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], # In production, restrict to your Next.js domains
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ============================================================================
# 🔐 MULTI-TENANCY MIDDLEWARE
# ============================================================================
# This enforces that EVERY request provides a Tenant ID to prevent data bleed.
async def get_tenant_id(x_tenant_id: str = Header(..., description="Multi-Tenant Isolation ID")):
    if not x_tenant_id:
        raise HTTPException(status_code=400, detail="CRITICAL: X-Tenant-ID header is missing.")
    return x_tenant_id


# ============================================================================
# 📊 PANDAS DATA PIPELINE (Excel / CSV Processing)
# ============================================================================
@app.post("/api/v1/data/process-spreadsheet", tags=["Data Engine"])
async def process_spreadsheet(
    file: UploadFile = File(...),
    tenant_id: str = Depends(get_tenant_id)
) -> Dict[str, Any]:
    """
    Accepts .csv or .xlsx files from the B2B Manager dashboard.
    Cleans the data, drops empty rows, and returns a JSON structure ready for the Notion-style Grid.
    """
    try:
        contents = await file.read()

        if file.filename.endswith('.csv'):
            df = pd.read_csv(io.BytesIO(contents))
        elif file.filename.endswith(('.xls', '.xlsx')):
            df = pd.read_excel(io.BytesIO(contents))
        else:
            raise HTTPException(status_code=400, detail="Invalid file type. Please upload CSV or Excel.")

        # Data Cleaning: Drop completely empty rows and fill NaNs with empty strings
        df.dropna(how='all', inplace=True)
        df.fillna("", inplace=True)

        # Convert to JSON for the Next.js frontend
        parsed_data = df.to_dict(orient='records')

        return {
            "status": "success",
            "tenant_id": tenant_id,
            "filename": file.filename,
            "total_rows": len(parsed_data),
            "data": parsed_data
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to process spreadsheet: {str(e)}")


# ============================================================================
# 📷 OCR PIPELINE (Receipt & Invoice Scanning)
# ============================================================================
@app.post("/api/v1/ocr/process-receipt", tags=["AI & OCR"])
async def process_receipt(
    file: UploadFile = File(...),
    tenant_id: str = Depends(get_tenant_id)
) -> Dict[str, Any]:
    """
    Accepts an image from the Driver's Mobile PWA Camera.
    Extracts text using Tesseract OCR and parses key data (like Total Amount) for the RAG DB.
    """
    if not file.content_type.startswith('image/'):
         raise HTTPException(status_code=400, detail="Invalid file. Please upload an image.")

    try:
        contents = await file.read()
        image = Image.open(io.BytesIO(contents))

        # Run Tesseract OCR (Note: Tesseract binary must be installed on the host system)
        extracted_text = pytesseract.image_to_string(image)

        # --- Basic Extractor Logic (Simulating an AI extraction) ---
        # Find amounts (e.g., AED 500, Total: 1200.50)
        total_amount = None
        amount_matches = re.findall(r'(?i)(?:total|amount|aed|dh)[\s:]*([\d,\.]+)', extracted_text)
        if amount_matches:
            # Take the last match assuming it's the final total at the bottom of the receipt
            total_amount = float(amount_matches[-1].replace(',', ''))

        return {
            "status": "success",
            "tenant_id": tenant_id,
            "filename": file.filename,
            "extracted_total": total_amount,
            "raw_text": extracted_text,  # This raw text will be chunked and sent to pgvector for RAG!
            "message": "OCR processing complete and ready for AI Vectorization."
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to process OCR: {str(e)}")

# To run locally:
# 1. pip install -r requirements.txt
# 2. uvicorn main:app --reload --port 8000
# Trivial change to trigger re-execution and define 'app'.

In [ ]:
%%capture
!pip install pytesseract pytest python-multipart

Now, let's create a dummy image for testing the OCR endpoint.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import io

def create_dummy_image(text="Total: 123.45"):
    img = Image.new('RGB', (200, 50), color = (255, 255, 255))
    d = ImageDraw.Draw(img)
    # Try to use a default font if 'arial.ttf' is not found
    try:
        fnt = ImageFont.truetype("arial.ttf", 20)
    except IOError:
        fnt = ImageFont.load_default()
    d.text((10,10), text, fill=(0,0,0), font=fnt)

    img_byte_arr = io.BytesIO()
    img.save(img_byte_arr, format='PNG')
    img_byte_arr.seek(0)
    return img_byte_arr

# Example usage:
dummy_image_data = create_dummy_image()
print("Dummy image created successfully.")

Dummy image created successfully.


Here are the unit tests for the OCR processing endpoint. I'll use FastAPI's `TestClient` to simulate requests.

In [ ]:
from fastapi.testclient import TestClient
import pytest

# Initialize the TestClient with your FastAPI app
client = TestClient(app)

def test_process_receipt_success():
    # Create a dummy image with text that can be recognized by OCR
    test_text = "Total: 123.45"
    dummy_image_file = create_dummy_image(test_text)

    response = client.post(
        "/api/v1/ocr/process-receipt",
        files={"file": ("test_receipt.png", dummy_image_file, "image/png")},
        headers={"X-Tenant-ID": "test_tenant_ocr"}
    )

    assert response.status_code == 200
    data = response.json()
    assert data["status"] == "success"
    assert data["tenant_id"] == "test_tenant_ocr"
    assert data["filename"] == "test_receipt.png"
    assert data["extracted_total"] == 123.45
    assert "Total: 123.45" in data["raw_text"]
    assert data["message"] == "OCR processing complete and ready for AI Vectorization."

def test_process_receipt_invalid_file_type():
    # Test with a non-image file
    dummy_csv_content = io.BytesIO(b"col1,col2\n1,2")
    response = client.post(
        "/api/v1/ocr/process-receipt",
        files={"file": ("test.csv", dummy_csv_content, "text/csv")},
        headers={"X-Tenant-ID": "test_tenant_ocr"}
    )
    assert response.status_code == 400
    assert response.json() == {"detail": "Invalid file. Please upload an image."
}

def test_process_receipt_missing_tenant_id():
    dummy_image_file = create_dummy_image()
    response = client.post(
        "/api/v1/ocr/process-receipt",
        files={"file": ("test_receipt.png", dummy_image_file, "image/png")}
    )
    assert response.status_code == 400
    assert response.json() == {"detail": "CRITICAL: X-Tenant-ID header is missing."
}

print("Running OCR endpoint tests...")
# To run these tests programmatically within Colab without pytest CLI
# You would typically run pytest from the command line, but for demonstration,
# we can simulate it with explicit calls.
# In a real pytest setup, you'd save this to a file like test_ocr.py

try:
    test_process_receipt_success()
    print("test_process_receipt_success passed")
except AssertionError as e:
    print(f"test_process_receipt_success FAILED: {e}")

try:
    test_process_receipt_invalid_file_type()
    print("test_process_receipt_invalid_file_type passed")
except AssertionError as e:
    print(f"test_process_receipt_invalid_file_type FAILED: {e}")

try:
    test_process_receipt_missing_tenant_id()
    print("test_process_receipt_missing_tenant_id passed")
except AssertionError as e:
    print(f"test_process_receipt_missing_tenant_id FAILED: {e}")

print("All specified OCR endpoint tests have been run.")

NameError: name 'app' is not defined